# Phase 2 smoke test on Colab (GPU)

Runs **one short** `curriculum_features` training on a free Colab GPU to confirm the Phase-2 configs actually train and write `loss.csv` / `val.csv` rows.

**Before you run:** Runtime → Change runtime type → **T4 GPU**.

This notebook clones **your working branch `curriculum-fixes`** (not `main`), where `phase2/` lives. Your two upstream deps (`TFM-Playground`, `tabicl`) are **not** in your GitHub repo, so it clones them separately **at the exact commits your local machine uses** — important because `curriculum/prior.py` imports a private path `tabicl.prior._prior_config`, which can move on upstream `main`.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Clone your repo + the two upstream deps (pinned)

In [ ]:
%%bash
set -e
cd /content
rm -rf ml-lab-curriculum

# Your repo, YOUR working branch (curriculum-fixes) — NOT main. phase2/ lives here.
BRANCH=curriculum-fixes
git clone --branch "$BRANCH" --single-branch https://github.com/parsafrei-droid/ml-lab-curriculum.git
cd ml-lab-curriculum
echo "on branch: $(git rev-parse --abbrev-ref HEAD)   (want: $BRANCH)"

# The two upstream deps, pinned to the SAME commits as your local checkout.
git clone https://github.com/automl/TFM-Playground.git
git -C TFM-Playground checkout 98e33be

git clone https://github.com/soda-inria/tabicl.git
git -C tabicl checkout 8f665ed

echo '--- layout ---'
ls -d curriculum scripts phase2 phase2/configs TFM-Playground tabicl 2>/dev/null || true

## 3. Safety net: recreate the 3 configs if they're somehow missing

`phase2/` is committed on `curriculum-fixes`, so the clone in cell 2 should already contain `phase2/configs`. This cell is just a fallback — it only writes anything if the configs are absent, otherwise it skips.

In [ ]:
import os, textwrap, pathlib
base = pathlib.Path('/content/ml-lab-curriculum/phase2/configs')
if (base / 'curriculum_features.yaml').exists():
    print('phase2/configs already present from the clone — skipping.')
else:
    base.mkdir(parents=True, exist_ok=True)
    (base / 'baseline_features.yaml').write_text(textwrap.dedent('''\
        name: baseline_features
        seed: 42
        epochs: 20
        steps: 100
        batch_size: 32
        lr: 0.003892
        num_datapoints: 200
        heads: 4
        embedding_size: 96
        hidden_size: 192
        layers: 3
        schedule:
          0: {min_features: 2, max_features: 60, max_classes: 10, noise_std: 0.3, num_layers: 6, hidden_dim: 128, num_causes: 12}
        '''))
    (base / 'curriculum_features.yaml').write_text(textwrap.dedent('''\
        name: curriculum_features
        seed: 42
        epochs: 20
        steps: 100
        batch_size: 32
        lr: 0.003892
        num_datapoints: 200
        heads: 4
        embedding_size: 96
        hidden_size: 192
        layers: 3
        schedule:
          0:    {min_features: 2, max_features: 4,  max_classes: 10, noise_std: 0.3, num_layers: 6, hidden_dim: 128, num_causes: 12}
          700:  {max_features: 20}
          1400: {max_features: 60}
        '''))
    print('wrote baseline_features.yaml + curriculum_features.yaml')
print(os.listdir(base))

## 4. Install dependencies (minimal set the smoke test actually imports)

`run.py` imports `tabicl` (prior) + `tfmplayground` (model/train). We install those two in editable mode plus `schedulefree` (the optimizer `train()` uses), and let `tabicl`'s own deps come along. We **skip** TFM-Playground's heavy pinned extras (`ticl`, `pfns`, `mlflow`, `torch==2.9`) — they aren't needed to train, and forcing `torch==2.9` fights Colab's preinstalled torch.

In [ ]:
%%bash
set -e
cd /content/ml-lab-curriculum
pip -q install schedulefree einops huggingface-hub 'scikit-learn>=1.5'
# tabicl provides the prior; install without letting it drag torch to a new version
pip -q install --no-deps -e ./tabicl
pip -q install --no-deps -e ./TFM-Playground
echo 'installs done'

## 5. Import sanity check

Confirms the private path resolves and the model builds **before** we spend time training. If this cell fails on `tabicl.prior._prior_config`, the pinned commit didn't take — re-run cell 2.

In [ ]:
import sys
for p in ['/content/ml-lab-curriculum',
          '/content/ml-lab-curriculum/TFM-Playground',
          '/content/ml-lab-curriculum/tabicl/src',
          '/content/ml-lab-curriculum/tabicl']:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
from tabicl.prior._prior_config import DEFAULT_SAMPLED_HP
from tfmplayground.models.nanotabpfn import NanoTabPFNModel
from tfmplayground.external_priors import TabICLPriorDataLoader
print('imports OK — private tabicl path resolves, model + prior import cleanly')

## 6. Run the smoke test (5 epochs = 500 steps, 1 seed)

Short on purpose — just to prove the config trains and writes rows. `--epochs 5` overrides the config's 20 (and run.py rescales the ramp thresholds automatically).

In [ ]:
%%bash
cd /content/ml-lab-curriculum
python scripts/run.py --config phase2/configs/curriculum_features.yaml \
    --epochs 5 --name smoke_curriculum_features_colab

## 7. Verify it actually produced data (the thing yesterday's run missed)

In [ ]:
import pandas as pd, pathlib
d = pathlib.Path('/content/ml-lab-curriculum/results/smoke_curriculum_features_colab')
loss = pd.read_csv(d / 'loss.csv')
val  = pd.read_csv(d / 'val.csv')
print('loss.csv rows:', len(loss)); display(loss)
print('val.csv rows :', len(val));  display(val)
assert len(loss) >= 1 and len(val) >= 1, 'SMOKE FAILED: no rows written — same problem as yesterday'
print('\nSMOKE PASSED — configs train and log. Ready for the full 2-arm x 3-seed 5k run.')